In [2]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path, final absolute url: d:\\Study\\Python\\llm-zoomcamp\\code
module_path = os.path.abspath(os.path.join('..', 'code'))

In [3]:
# append the code folder in sys.path if not already exists (100% likely)
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
sys.path

In [4]:
from sqlite_ingest import load_faq_data,build_index_persist_init

documents=load_faq_data()
print(f"Loaded {len(documents)} documents")
build_index_persist_init(documents)

Loaded 1208 documents


In [ ]:
#import importlib
#importlib.reload(sqlite_ingest)

<module 'sqlite_ingest' from 'd:\\Study\\Python\\llm-zoomcamp\\code\\sqlite_ingest.py'>

In [ ]:
# if you only want to update index by adding couple additional documents
import time
from sqlitesearch import TextSearchIndex

docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="../database/faq.db"
)

# add a doc with same id will overwrite instead of creating duplications
for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

In [14]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="../database/faq.db"
)

results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['The course has already started. Can I still join it?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'Course - Can I still join the course after the start date?',
 'Course: Can I still join the course after the start date?']

In [15]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [16]:
answer = assistant.rag('Can I still join the course after it started?')
print(answer)

Yes, you can still join after it started. If you want a certificate, you need to submit your project while submissions are still open.


In [17]:
sqlite_index.close()